# Query Multilayer Networks with the DSL

This notebook demonstrates py3plex's SQL-like Domain-Specific Language (DSL) for querying multilayer networks.

## Why Use the DSL?

* **Graph-aware**: Understands multilayer structures, layers, and (node, layer) tuple semantics
* **Type-safe**: Builder API with IDE autocompletion
* **Integrated**: Compute centrality and metrics directly in queries
* **Flexible**: String syntax for quick prototyping, builder API for production

## Installation

In [ ]:
!pip install py3plex -q

## Setup: Create a Sample Network

Let's create a multilayer network with social and work layers.

In [ ]:
from py3plex.core import multinet

# Create sample network
network = multinet.multi_layer_network()

# Add edges across layers
network.add_edges([
    # Social layer
    ['Alice', 'social', 'Bob', 'social', 1],
    ['Bob', 'social', 'Charlie', 'social', 1],
    ['Charlie', 'social', 'David', 'social', 1],
    ['Alice', 'social', 'Eve', 'social', 1],
    ['David', 'social', 'Frank', 'social', 1],
    # Work layer
    ['Alice', 'work', 'Bob', 'work', 1],
    ['Bob', 'work', 'Charlie', 'work', 1],
    ['Charlie', 'work', 'Frank', 'work', 1],
    # Hobby layer
    ['Alice', 'hobby', 'David', 'hobby', 1],
    ['Eve', 'hobby', 'Frank', 'hobby', 1],
], input_type="list")

print("Network created:")
network.basic_stats()

## Method 1: String Syntax (Quick and Readable)

Use SQL-like strings for quick queries.

In [ ]:
from py3plex.dsl import execute_query

# Get all nodes
result = execute_query(network, 'SELECT nodes')
print(f"Found {len(result)} nodes")

# Get nodes from social layer only
result = execute_query(network, 'SELECT nodes FROM layer="social"')
print(f"\nSocial layer has {len(result)} nodes")

# Filter by degree
result = execute_query(network, 'SELECT nodes WHERE degree > 1')
df = result.to_pandas()
print("\nNodes with degree > 1:")
print(df)

## Method 2: Builder API (Type-Safe, Chainable)

Use the Python builder API for type safety and IDE autocompletion.

In [ ]:
from py3plex.dsl import Q, L

# Simple query
result = Q.nodes().execute(network)
print(f"Total nodes: {len(result)}")

# Query specific layer
result = (
    Q.nodes()
     .from_layers(L["social"])
     .execute(network)
)
print(f"\nSocial layer nodes: {len(result)}")

# Filter and compute
result = (
    Q.nodes()
     .from_layers(L["*"])  # All layers
     .where(degree__gt=1)  # Django-style filter
     .compute("degree", "betweenness_centrality")
     .order_by("-degree")  # Sort descending
     .execute(network)
)

df = result.to_pandas()
print("\nHigh-degree nodes with centrality:")
print(df)

## Layer Algebra: Combine Layers

Use operators to combine layers: `+` (union), `&` (intersection), `-` (difference)

In [ ]:
# Union: nodes from social OR work
result = (
    Q.nodes()
     .from_layers(L["social"] + L["work"])
     .execute(network)
)
print(f"Nodes in social + work: {len(result)}")

# All layers
result = (
    Q.nodes()
     .from_layers(L["*"])
     .execute(network)
)
print(f"Nodes in all layers: {len(result)}")

## Compute Multiple Metrics

Calculate various centrality measures in a single query.

In [ ]:
result = (
    Q.nodes()
     .from_layers(L["*"])
     .compute(
         "degree",
         "betweenness_centrality",
         "closeness_centrality",
         "pagerank"
     )
     .order_by("-betweenness_centrality")
     .limit(10)
     .execute(network)
)

df = result.to_pandas()
print("Top 10 nodes by betweenness centrality:")
print(df)

## Filter with Complex Conditions

Use Django-style lookups for flexible filtering.

In [ ]:
# Nodes with degree greater than 1 in the social layer
result = (
    Q.nodes()
     .from_layers(L["social"])
     .where(degree__gt=1)
     .compute("degree")
     .execute(network)
)

df = result.to_pandas()
print("Social layer hubs (degree > 1):")
print(df)

# Nodes NOT in a specific layer
result = (
    Q.nodes()
     .where(layer__ne="hobby")
     .compute("degree")
     .execute(network)
)

df = result.to_pandas()
print(f"\nNodes not in hobby layer: {len(df)}")

## Query Edges

Query edges within or between layers.

In [ ]:
# Get all edges
result = Q.edges().execute(network)
print(f"Total edges: {len(result)}")

# Edges from social layer only
result = (
    Q.edges()
     .from_layers(L["social"])
     .execute(network)
)
print(f"\nEdges in social layer: {len(result)}")

# View edge data
df = result.to_pandas()
print("\nSample edges:")
print(df.head())

## Export Results

Export query results to various formats.

In [ ]:
result = (
    Q.nodes()
     .from_layers(L["*"])
     .compute("degree", "betweenness_centrality")
     .execute(network)
)

# Export to pandas DataFrame
df = result.to_pandas()
print("Pandas DataFrame:")
print(df.head())

# Export to dictionary
data_dict = result.to_dict()
print(f"\nDictionary with {len(data_dict)} entries")

# Export to NetworkX graph
nx_graph = result.to_networkx()
print(f"\nNetworkX graph: {nx_graph.number_of_nodes()} nodes, {nx_graph.number_of_edges()} edges")

## Group Results by Layer

Use `per_layer()` and `per_layer_pair()` to group results by layers.

In [ ]:
# Group results by layer for per-layer analysis
# This is useful for comparing layers or finding top nodes in each layer
result = (
    Q.nodes()
     .from_layers(L["*"])
     .compute("degree", "betweenness_centrality")
     .per_layer()  # Start layer-wise grouping
        .top_k(3, "degree")  # Get top 3 nodes per layer
     .end_grouping()  # End layer-wise grouping
     .execute(network)
)

print("\nTop 3 nodes per layer by degree:")
print(result.to_pandas())

# Get structured grouping metadata and summary
print("\nGrouping summary (aggregated by layer):")
summary = result.group_summary()  # Returns DataFrame with group statistics
print(summary)

## Uncertainty Quantification

Compute **multiple** centrality metrics with confidence intervals using bootstrap resampling.

In [ ]:
from py3plex.dsl import UQ

# Uncertainty Quantification (UQ) using bootstrap resampling
# Compute MULTIPLE centrality metrics with confidence intervals
result = (
    Q.nodes()
     .from_layers(L["*"])
     .uq(method="bootstrap", n_samples=50, ci=0.95)  # 50 bootstrap samples, 95% CI
     .compute(
         "degree",
         "betweenness_centrality",
         "closeness_centrality",
         "eigenvector_centrality"
     )
     .order_by("-betweenness_centrality")  # Sort by mean betweenness
     .limit(5)  # Top 5 nodes
     .execute(network)
)

# Convert to DataFrame with uncertainty columns
# expand_uncertainty=True adds _std, _ci95_low, _ci95_high columns for each metric
df = result.to_pandas(expand_uncertainty=True)

print("\nTop 5 nodes with MULTIPLE centrality metrics and uncertainty:")
print("\n1. Betweenness Centrality:")
print(df[['id', 'layer', 'betweenness_centrality', 
          'betweenness_centrality_std', 'betweenness_centrality_ci95_low', 
          'betweenness_centrality_ci95_high']])

print("\n2. Degree Centrality:")
print(df[['id', 'degree', 'degree_std', 'degree_ci95_low', 'degree_ci95_high']])

print("\n3. Closeness Centrality:")
print(df[['id', 'closeness_centrality', 'closeness_centrality_std',
          'closeness_centrality_ci95_low', 'closeness_centrality_ci95_high']])

# Compare centrality rankings stability
print("\n4. Ranking Stability (Coefficient of Variation):")
df['betweenness_cv'] = df['betweenness_centrality_std'] / df['betweenness_centrality']
df['degree_cv'] = df['degree_std'] / df['degree']
print(df[['id', 'betweenness_cv', 'degree_cv']])
print("\nNote: Lower CV = more stable/reliable centrality ranking")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualize centrality uncertainty with error bars
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Betweenness centrality with error bars
ax = axes[0, 0]
x = range(len(df))
ax.errorbar(x, df['betweenness_centrality'], 
            yerr=df['betweenness_centrality_std'],
            fmt='o-', capsize=5, capthick=2, linewidth=2, markersize=8)
ax.set_xticks(x)
ax.set_xticklabels(df['id'], rotation=45, ha='right')
ax.set_ylabel('Betweenness Centrality')
ax.set_title('Betweenness Centrality with ±1 Std Dev')
ax.grid(True, alpha=0.3)

# Plot 2: Degree centrality with error bars
ax = axes[0, 1]
ax.errorbar(x, df['degree'], 
            yerr=df['degree_std'],
            fmt='s-', color='orange', capsize=5, capthick=2, linewidth=2, markersize=8)
ax.set_xticks(x)
ax.set_xticklabels(df['id'], rotation=45, ha='right')
ax.set_ylabel('Degree')
ax.set_title('Degree with ±1 Std Dev')
ax.grid(True, alpha=0.3)

# Plot 3: Confidence interval ranges
ax = axes[1, 0]
ci_widths = df['betweenness_centrality_ci95_high'] - df['betweenness_centrality_ci95_low']
ax.barh(range(len(df)), ci_widths, color='steelblue')
ax.set_yticks(range(len(df)))
ax.set_yticklabels(df['id'])
ax.set_xlabel('95% CI Width')
ax.set_title('Betweenness Centrality: Confidence Interval Width')
ax.grid(True, alpha=0.3, axis='x')

# Plot 4: Coefficient of Variation comparison
ax = axes[1, 1]
width = 0.35
x_pos = np.arange(len(df))
ax.bar(x_pos - width/2, df['betweenness_cv'], width, label='Betweenness', color='steelblue')
ax.bar(x_pos + width/2, df['degree_cv'], width, label='Degree', color='orange')
ax.set_xticks(x_pos)
ax.set_xticklabels(df['id'], rotation=45, ha='right')
ax.set_ylabel('Coefficient of Variation')
ax.set_title('Centrality Stability (Lower = More Stable)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0.2, color='red', linestyle='--', linewidth=1, alpha=0.5, label='Stability threshold')

plt.tight_layout()
plt.show()

print("\nKey Insights:")
print("- Error bars show the uncertainty in each metric")
print("- Wider CI = less reliable metric for that node")
print("- CV < 0.2 indicates stable/reliable centrality ranking")

## Temporal Queries

Query temporal networks with time windows (requires temporal network).

In [ ]:
from py3plex.core.temporal_multinet import TemporalMultiLayerNetwork

# Create a temporal network for demonstration
temporal_net = TemporalMultiLayerNetwork()
temporal_net.add_edge('A', 'B', layer='social', time=1.0)
temporal_net.add_edge('B', 'C', layer='social', time=2.0)
temporal_net.add_edge('A', 'C', layer='social', time=3.0)
temporal_net.add_edge('D', 'E', layer='social', time=4.0)

print("\nTemporal network created with 4 edges at different times")

# Query with time window
result = (
    Q.nodes()
     .from_layers(L["social"])
     .window(start=1.0, end=3.0)  # Only events between t=1.0 and t=3.0
     .compute("degree")
     .execute(temporal_net)
)

print("\nNodes active in time window [1.0, 3.0]:")
print(result.to_pandas())

## Summary

In this tutorial, you learned:

* ✅ String DSL syntax for quick queries
* ✅ Builder API (Q, L) for type-safe queries
* ✅ Layer algebra for combining layers
* ✅ Computing multiple metrics
* ✅ Complex filtering conditions
* ✅ Edge queries and analysis
* ✅ Grouping results by layer
* ✅ Uncertainty quantification with bootstrap
* ✅ Temporal queries with time windows
* ✅ Export options (pandas, JSON, CSV)

## Next Steps

* Check the [DSL reference](https://skblaz.github.io/py3plex/reference/dsl.html)
* Try [dynamics simulation](simulate_dynamics.ipynb)
* Explore [community detection](community_detection.ipynb)
* Read the [full documentation](https://skblaz.github.io/py3plex/)